# RT-DETR on DocLayNet — Kaggle training run

Before running anything, in the panel on the right:

1. **Settings -> Accelerator -> GPU T4 x2** (the script uses one of them)
2. **Settings -> Internet -> On** (needed to pull the dataset and the weights)

Then use **Save Version -> Save & Run All (Commit)** rather than running
interactively. A committed run keeps going after I close the browser tab, which
matters because this takes a few hours and I do not want to lose it to a dropped
connection.

## 1. Confirm what hardware I actually got

Kaggle does not always hand you the accelerator you selected, and "training took
3 hours" means nothing without knowing what it ran on. This value also gets
written into `run_metadata.json` by the training script so the reproducibility
claim in my README is backed by something.

In [ ]:
!nvidia-smi

## 2. Install

Ultralytics brings its own torch, which Kaggle already has, so this is quick.
I pin ultralytics because RT-DETR's training arguments have moved between
minor versions and I want the reviewers running the same code path I did.

In [ ]:
!pip install -q ultralytics==8.3.40 "datasets<4.0.0"


## 3. Pull the repo

Replace the URL with my repo. Everything below calls the scripts in it rather
than redefining logic in the notebook - I did not want a situation where the
notebook and the repo disagree about how the data was prepared.

In [ ]:
REPO_URL = "https://github.com/RISHIVELS/rap-doclayout.git"

# %cd into a directory this cell is about to rm -rf breaks the shell's
# own working directory on the *next* run of this cell (getcwd fails
# because the path it thinks it is standing in no longer exists). So I
# step out to a stable parent directory before deleting anything.
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1


## 4. Build the dataset

This downloads ~3.8 GB on the first run and converts it into the image/label
layout Ultralytics expects.

The `--verify 12` flag renders twelve annotated pages so I can look at them in
the next cell. That check is not optional for me - see the note under it.

In [ ]:
!python scripts/prepare_dataset.py --out /kaggle/working/data/doclaynet --verify 12

## 5. STOP. Look at the labels before training.

This is the gate. My class ordering is 0-indexed alphabetical, which I inferred
from the data rather than found stated unambiguously anywhere. If it is off by
one, `Table` becomes `Section-header` across the whole dataset — and nothing
breaks. Training succeeds, the loss falls, the metrics look plausible, and every
number and failure case I write up afterwards describes a model that learned
something other than what I claim it learned.

No unit test can catch this, because the data is perfectly self-consistent under
either assumption. The only check that works is looking.

**What I am checking:** is the box labelled `Table` actually drawn around a
table? Is `Picture` around a figure? If not, stop and fix the index base before
spending GPU quota.

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

checks = sorted(Path("/kaggle/working/assets/label_check").glob("*.png"))
print(f"{len(checks)} pages to check\n")

for path in checks[:4]:
    display(Image.open(path).resize((620, 620)))

## 6. Train

RT-DETR-L, 30 epochs, batch 8 at 640px, AMP on.

Sizing notes for the T4 (16 GB): batch 8 with AMP is what fits comfortably at
this resolution. I would have preferred 800px or 1024px input — thin classes
like `Footnote` and `Page-footer` lose a lot of detail when a 1025px page is
squeezed to 640 — but that trade was forced by the time budget, and I would
rather report the limitation honestly than run out of session.

Checkpointing every epoch, so a lost session costs one epoch rather than the run.

In [ ]:
!python scripts/train.py \
    --data /kaggle/working/data/doclaynet/doclaynet.yaml \
    --epochs 30 \
    --batch 8 \
    --imgsz 640 \
    --device 0 \
    --name rtdetr_doclaynet

## 7. Evaluate on the held-out test split

Overall mAP, per-class AP, the confusion matrix, per-document-category
breakdown, and the query-saturation rate. The per-category split is the one I
care most about: a single aggregate mAP cannot tell me whether the model learned
document structure or just learned what annual reports look like.

In [ ]:
!python scripts/evaluate.py \
    --weights runs/detect/rtdetr_doclaynet/weights/best.pt \
    --data /kaggle/working/data/doclaynet/doclaynet.yaml \
    --out reports

## 8. Mine the failure cases

Ranks every test image by error and dumps the worst ones as side-by-side
ground-truth/prediction renders. The five failure cases in my memo are picked
from these, so they are evidence rather than things I imagined my model might
get wrong.

In [ ]:
!python scripts/mine_failures.py \
    --weights runs/detect/rtdetr_doclaynet/weights/best.pt \
    --data /kaggle/working/data/doclaynet \
    --out reports/failures \
    --top 25

## 9. Collect the outputs

Everything I need off this machine: the weights, the metrics, the run metadata
and the failure renders.

In [ ]:
import shutil
from pathlib import Path

run = Path("runs/detect/rtdetr_doclaynet")
out = Path("/kaggle/working/submission")
out.mkdir(exist_ok=True)

shutil.copy(run / "weights/best.pt", out / "best.pt")
shutil.copy(run / "run_metadata.json", out / "run_metadata.json")
if Path("reports").exists():
    shutil.copytree("reports", out / "reports", dirs_exist_ok=True)

for path in sorted(out.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:8.1f} MB  {path.relative_to(out)}")